# Basic design of RF systems — A: hadron synchrotron

Current tutors: S. Albright, H. Damerau, A. Lasheen, M. Taquet

Contributors: J. Flowerdew, L. Intelisano, D. Quartullo, F. Tecker, C. Völlinger, L. Valle, M. Zampetakis

## Links

- Introductory CAS website: https://indico.cern.ch/event/1622828/
- Programme of the CAS: https://indico.cern.ch/event/1622828/attachments/3196386/5990113/Timetable_Introductory2026_ver1.6b.pdf
- Python software installation for longitudinal exercises: https://github.com/cerncas/hands-on-longitudinal-exercises/blob/main/README.md
- Longitudinal hands-on, link to content and cheat sheets: https://indico.cern.ch/event/1622828/contributions/7202974/

# Introduction

This hands-on session exists in **two versions**, please choose **one** according to your interest:
- **Version A**: design the RF system for a proton synchrotron, the upgrade of the SPS to a hadron injector for the Future Circular Collider (FCC-hh)
- **Version B**: develop the RF system for a hypothetical beam energy and current upgrade of an electron storage ring (Soleil)

**This notebook is version A: hadron synchrotron.**

The *Lecture slides* lines below point to the slides of the **Longitudinal Dynamics** lecture (F. Tecker) and of the **RF Systems** lecture (C. Völlinger) at this school, where the corresponding topics are introduced. The numbers are the slide numbers printed at the bottom of the slides.

Each question is followed by an empty code cell for your calculation. The given values are defined once, in the cell below the parameter table.

## Import modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as sciCont

# Upgrade the CERN-SPS to higher energy as a hadron injector for the Future Circular Collider (FCC-hh)

### Design an RF system optimized for the Superconducting Super Proton Synchrotron (scSPS)
(see, e.g. <a href="https://indico.cern.ch/event/591312/contributions/2386529/attachments/1379193/2133450/scSPS_Note_v2.pdf">F. Burkart et al., SPS energy upgrade considerations</a>)

In this hands-on session we will design the RF system for a proton synchrotron.

The goal of the session is to calculate the relevant longitudinal parameters.

The notebook is constructed with the following purpose in mind:

1. Design a linear acceleration cycle in bending field which determines average energy gain per turn and stable phase.
2. Knowing energy gain per turn and the number of particles accelerated, the average RF power to the beam is calculated.
3. Injection and extraction energies also fix the revolution frequencies and, in combination with the harmonic number, define the RF frequency range.
4. The acceleration of a bunch with a given size in the longitudinal phase space, i.e. longitudinal emittance, requires a sufficient bucket area. This area is defined by the RF voltage. Interesting effects occur around the energy of transition crossing, which also deserves a more detailed look.
5. The energy transfer does not only take place from the cavity to the beam, but also the beam may induce significant voltage back into the cavity. This phenomenon has been introduced as beam loading.
6. New RF systems for particle accelerators are rarely designed from scratch, but inspired by existing installations. Compare your RF system design with the existing one of the CERN SPS, as well as with the RF system of the now-decommissioned Tevatron, or any other RF system you may find.

**Basic parameters of the superconducting Proton Synchrotron (scSPS) at CERN**

| Parameter                        |                                                                     |
| -------------------------------- | ------------------------------------------------------------------- |
| Kinetic energy at injection      | $E_\mathrm{inj} = 13.1\,\mathrm{GeV}$                               |
| Kinetic energy at extraction     | $E_\mathrm{ext} = 1300\,\mathrm{GeV}$                               |
| Circumference                    | $2 \pi R = 6911.5\,\mathrm{m}$                                      |
| Bending radius                   | $\rho = 741.3\,\mathrm{m}$                                          |
| Transition gamma                 | $\gamma_\mathrm{tr} = 18$                                           |
| Acceleration time                | $4\,\mathrm{s}$                                                     |
| Longitudinal emittance per bunch | $\varepsilon_\mathrm{l} = 0.4\,\mathrm{eVs}\ldots0.5\,\mathrm{eVs}$ |
| Maximum bucket filling factor    | $\varepsilon_\mathrm{l}/A_\mathrm{bucket} = 0.8$                    |
| Total beam intensity             | $N = 1 \cdot 10^{13} \,\mathrm{protons}$                            |
| Minimum bunch spacing            | $25 \,\mathrm{ns}$                                                  |

In [ ]:
# Physical constants
c0 = sciCont.c
e0 = sciCont.e
protonMass = sciCont.physical_constants["proton mass energy equivalent in MeV"][0] * 1e6
protonCharge = 1  # e

# Given values, from the table above
injectionEnergy = 13.1e9  # eV
extractionEnergy = 1.3e12  # eV
circumference = 2 * np.pi * 1100  # m
bendingRadius = 741.3  # m
accelerationDuration = 4  # s
gamma_T = 18
longitudinalEmittance = 0.45  # eVs
targetFillingFactor = 0.8
nProtons = 1e13

## Exercise 1: Average energy gain and stable phase

*Lecture slides: Tecker 4, 18, 20*

1. How much energy does the particle gain during each turn assuming a constant ramp rate in bending field, $B$?
    - *Assume a linear acceleration ramp from $B_\mathrm{inj}$ to $B_\mathrm{ext}$ with $dB/dt = \mathrm{const.}$*
    - *Convert the given energies, $E = m_0 \gamma c^2$, at injection and extraction to momenta, $p = m_0 \beta \gamma c$, with $\beta = v/c = \sqrt{1 - 1/\gamma^2}$ or $\gamma = m/m_0 = 1/\sqrt{1-\beta^2}$. Note that the common unit for energy is eV, and the unit for momentum is eV/c, hence also the particle mass is most handy in units of eV.* *(Tecker 4)*
    - *Use the momentum to calculate the required bending field at injection and extraction.* *(Tecker 18)*
    - *Assuming a linear ramp, the ramp rate is just given by extraction minus injection field, divided by the acceleration time.*
    - *Derive the average energy gain per turn from the ramp rate:  $\Delta E_\mathrm{turn} = 2 \pi q R_\mathrm{mean} \rho \cdot dB/dt$* *(Tecker 20)*

In [ ]:
# --- Solution ---
momentumInjection = np.sqrt((protonMass + injectionEnergy) ** 2 - protonMass**2)
momentumExtraction = np.sqrt((protonMass + extractionEnergy) ** 2 - protonMass**2)

print("Momentum at injection:  " + str(momentumInjection / 1e9) + " GeV/c")
print("Momentum at extraction: " + str(momentumExtraction / 1e9) + " GeV/c")
magneticFieldInjection = momentumInjection / (c0 * bendingRadius * protonCharge)
magneticFieldExtraction = momentumExtraction / (c0 * bendingRadius * protonCharge)
averageBDot = (magneticFieldExtraction - magneticFieldInjection) / accelerationDuration

print("Average B-Dot: " + str(averageBDot) + " T/s")
averageEnergyGainPerTurn = (
    2 * np.pi * protonCharge * circumference / (2 * np.pi) * bendingRadius * averageBDot
)
print("Average per turn energy gain: " + str(averageEnergyGainPerTurn / 1e6) + " MeV")

2. What would the stable phase be for an RF voltage of 20 MV?
    - *The bunch must be placed at a stable phase such that the RF voltage exactly matches required energy gain per turn: $\sin \phi_\mathrm{S} = \Delta E / V_\mathrm{RF}$* *(Tecker 20, 31)*

In [ ]:
# --- Solution ---
rfVoltage = 20000e3  # V
stablePhase = np.arcsin(averageEnergyGainPerTurn / rfVoltage)
print("Stable phase angle: " + str(180 / np.pi * stablePhase) + " deg")

print("Stable phase angle above transition: " + str(180 - 180 / np.pi * stablePhase) + " deg")

# Answer: 21.8 deg is the stable phase below transition, as at injection here (gamma = 15 < gamma_t = 18).
# Above transition, i.e. during almost all of the ramp, the stable phase is 180 deg minus that value

## Exercise 2: Power transfer to the beam

1. How much power is required from the RF system to accelerate the beam?
    - *Keep in mind that power is nothing but a change of energy within a given time, the duration of the acceleration.*
    - *The energy gain of the whole beam is the gain per particle times the number of particles, to be converted from eV to joules with the elementary charge.*

In [ ]:
# --- Solution ---
energyGainJoules = (extractionEnergy - injectionEnergy) * e0 * nProtons
averagePowerToBeam = energyGainJoules / accelerationDuration
print("Average power to beam: " + str(averagePowerToBeam / 1e3) + " kW")

## Exercise 3: RF frequency and harmonic

*Lecture slides: Tecker 5, 15, 21 · Völlinger 9–13*

1. The bunch spacing of $25\,\mathrm{ns}$ corresponds to a bunch frequency of about $40\,\mathrm{MHz}$ at flat-top. What is the corresponding harmonic number?
    - *Calculate the revolution frequencies at injection and extraction.* *(Tecker 5, 21)*
    - *The harmonic number is an integer: round it, and recalculate the RF frequency from the revolution frequency.*

In [ ]:
# --- Solution ---
betaInjection = np.sqrt(1 - (protonMass / (protonMass + injectionEnergy)) ** 2)
betaExtraction = np.sqrt(1 - (protonMass / (protonMass + extractionEnergy)) ** 2)

print(f"beta injection: {betaInjection}, beta extraction: {betaExtraction}")
revolutionFrequencyInjection = c0 * betaInjection / circumference
revolutionFrequencyExtraction = c0 * betaExtraction / circumference

print("Revolution frequency at injection:  " + str(revolutionFrequencyInjection / 1e3) + "kHz")
print("Revolution frequency at extraction: " + str(revolutionFrequencyExtraction / 1e3) + "kHz")
flatTopRFFrequencyApproximate = 1 / 25e-9
harmonicNumber = flatTopRFFrequencyApproximate / revolutionFrequencyExtraction
print("40 MHz/revolution frequency at extraction: " + str(harmonicNumber))
harmonicNumber = round(harmonicNumber)
print("Nearest integer harmonic number, h = " + str(harmonicNumber))

2. Calculate the exact RF frequency at injection and extraction.
    - *Derive the revolution and RF frequency swing. The difference between lowest and highest frequency of an RF cavity is also referred to as frequency swing or tuning range.*

In [ ]:
# --- Solution ---
injectionRFFrequency = harmonicNumber * revolutionFrequencyInjection
extractionRFFrequency = harmonicNumber * revolutionFrequencyExtraction

frequencySwingFRev = revolutionFrequencyExtraction - revolutionFrequencyInjection
frequencySwingRF = extractionRFFrequency - injectionRFFrequency

print("RF frequency at injection:  " + str(injectionRFFrequency / 1e6) + " MHz")
print("RF frequency at extraction: " + str(extractionRFFrequency / 1e6) + " MHz")
print(
    "Revolution frequency swing: "
    + str(frequencySwingFRev / 1e3)
    + " kHz ("
    + str(100 * frequencySwingFRev / revolutionFrequencyInjection)
    + " %)"
)
print("RF frequency swing: " + str(frequencySwingRF / 1e3) + " kHz")

3. The designers of the SPS chose a harmonic number of h = 4620 to use commercial power sources at $200\,\mathrm{MHz}$. *(Völlinger 12–13)* Calculate the exact RF frequency at injection and extraction in that case.

In [ ]:
# --- Solution ---
harmonicNumber = 4620
injectionRFFrequency = harmonicNumber * revolutionFrequencyInjection
extractionRFFrequency = harmonicNumber * revolutionFrequencyExtraction

frequencySwingFRev = revolutionFrequencyExtraction - revolutionFrequencyInjection
frequencySwingRF = extractionRFFrequency - injectionRFFrequency

print("RF frequency at injection:  " + str(injectionRFFrequency / 1e6) + " MHz")
print("RF frequency at extraction: " + str(extractionRFFrequency / 1e6) + " MHz")
print(
    "Revolution frequency swing: "
    + str(frequencySwingFRev / 1e3)
    + " kHz ("
    + str(100 * frequencySwingFRev / revolutionFrequencyInjection)
    + " %)"
)
print("RF frequency swing: " + str(frequencySwingRF / 1e3) + " kHz")

## Exercise 4: Calculate bucket area during the cycle, determine RF voltage along the cycle

*Lecture slides: Tecker 22, 32, 41, 43, 56*

1. Plot momentum, kinetic energy and revolution frequency (and/or further parameters) during the cycle.
    - *The bending field rises linearly in time: build it as an array with `np.linspace`, then momentum, energy and revolution frequency follow as arrays, with the same relations as in Exercise 1.*

In [ ]:
# --- Solution ---

timeRange = np.linspace(0, accelerationDuration, 1000)
BFieldRange = np.linspace(magneticFieldInjection, magneticFieldExtraction, len(timeRange))
momentumRange = protonCharge * bendingRadius * BFieldRange * c0

plt.figure()
plt.plot(timeRange, momentumRange / 1e9)
plt.xlabel("Cycle Time (s)")
plt.ylabel("Beam Momentum (GeV/c)")
plt.show()

energyRange = np.sqrt(momentumRange**2 + protonMass**2)

plt.figure()
plt.plot(timeRange, (energyRange - protonMass) / 1e9)
plt.xlabel("Cycle Time (s)")
plt.ylabel("Beam Kinetic Energy (GeV)")
plt.show()

betaRange = momentumRange / energyRange
gammaRange = 1 / np.sqrt(1 - betaRange**2)

fRevRange = c0 * betaRange / circumference

plt.figure()
plt.plot(timeRange, fRevRange / 1e3)
plt.xlabel("Cycle Time (s)")
plt.ylabel("Revolution Frequency (kHz)")
plt.show()

2. Calculate and plot the bucket area along the cycle and choose an RF voltage such that a bunch with $0.45\,\mathrm{eVs}$ longitudinal emittance can be comfortably accelerated, e.g. $\varepsilon_\mathrm{l}/A_\mathrm{bucket} \simeq 0.8$, corresponding to a bucket area of about $0.56\,\mathrm{eVs}$.
    - *Define an auxiliary function for the bucket area reduction due to the energy loss per turn, applying the approximation (S. Y. Lee book, p. 242) $\alpha(\phi_\mathrm{S}) \simeq \cfrac{1 - \sin \phi_\mathrm{S}}{1 + \sin \phi_\mathrm{S}}$.* *(Tecker 56)*
    - *Assume a harmonic number of $h = 4620$.*
    - *Use the bucket area of the cheat sheet, with the energy and the phase slip factor as arrays along the cycle.*

In [ ]:
# --- Solution ---
targetBucketArea = longitudinalEmittance / targetFillingFactor

print("Target bucket area: " + str(targetBucketArea) + " eVs")


def reduction_ratio(deltaE, voltage):
    phis = np.arcsin(deltaE / voltage)
    return (1 - np.sin(phis)) / (1 + np.sin(phis))


rfHarmonic = 4620
rfVoltage = 20e6  # V

timeRange = np.linspace(0, accelerationDuration, 1000)
BFieldRange = np.linspace(magneticFieldInjection, magneticFieldExtraction, len(timeRange))
momentumRange = protonCharge * bendingRadius * BFieldRange * c0
energyRange = np.sqrt(momentumRange**2 + protonMass**2)
gammaRange = 1 / np.sqrt(1 - betaRange**2)
phaseSlipFactor = 1 / gamma_T**2 - 1 / gammaRange**2

reductionFactor = reduction_ratio(averageEnergyGainPerTurn, rfVoltage)

bucketAreaRange = (
    4
    / c0
    * circumference
    * np.sqrt((2 * energyRange * rfVoltage) / (np.pi**3 * rfHarmonic**3 * np.abs(phaseSlipFactor)))
    * reductionFactor
)

plt.figure()
plt.plot(timeRange, bucketAreaRange)
plt.axhline(targetBucketArea, linestyle="--")
plt.ylim(0, 5 * targetBucketArea)
plt.xlabel("Cycle Time (s)")
plt.ylabel("Bucket Area (eVs)")
plt.show()

# larger at injection than in this linear-ramp model

3. Zoom around transition crossing. What happens there? *(Tecker 32)*

In [ ]:
# --- Solution ---
plt.figure()
plt.plot(timeRange, bucketAreaRange)
plt.axhline(targetBucketArea, linestyle="--")
plt.ylim(0, 5 * targetBucketArea)
plt.xlim(0, 0.1)
plt.xlabel("Cycle Time (s)")
plt.ylabel("Bucket Area (eVs)")
plt.show()

# jumps from phi_s to 180 deg - phi_s when crossing transition, i.e. the RF phase has to jump by 180 deg - 2 phi_s

## Exercise 5: Requirements for beam loading

*Lecture slides: Völlinger 72–73, 77–79*

1. Assume an RF cavity resonator with an $R/Q$ of about $100\,\mathrm{\Omega}$. *(Völlinger 72–73)* What is the beam induced voltage and power due to the passage of one bunch? *(Völlinger 78)*
    - *For the sake of the exercise, we assume here that the whole beam is concentrated in one bunch.*
    - *The voltage induced by a charge passing the cavity is given in the cheat sheet.*
    - *The power is the induced voltage times the beam current, and the beam current is the total charge times the revolution frequency.*

In [ ]:
# --- Solution ---
harmonicNumber = rfHarmonic
RUponQ = 100
VInduced = nProtons * e0 * RUponQ * harmonicNumber * revolutionFrequencyExtraction * 2 * np.pi

print("Beam loading induced voltage (flat top): " + str(VInduced / 1e3) + " kV")

beamCurrent = e0 * nProtons * revolutionFrequencyExtraction
beamLoadingPower = VInduced * beamCurrent

print("Beam loading power (flat top): " + str(beamLoadingPower / 1e3) + " kW")

2. Under which circumstances do you really need that power?

In [ ]:
# --- Solution ---
# Answer: This additional power would be needed to fully compensate beam loading and operate the cavity at any phase

## Exercise 6: Comparison with the RF systems of the present SPS and of the Fermilab Tevatron

*Lecture slides: Völlinger 13, 97–100*

1. Compare the parameters of your RF system with the ones of the present SPS and the Tevatron. Exchange with the tutors on how the different RF systems compare with one another.

| Parameter                                       | Unit | scSPS         | SPS       | Tevatron    |
| ----------------------------------------------- | ---- | ------------- | --------- | ----------- |
| Beam energy, $E_\mathrm{kin}$                   | GeV  | 13.1 to 1300  | 14 to 450 | 120 to 980  |
| Circumference, $2\pi R$                         | km   | 6.9           | 6.9       | 6.3         |
| Bending radius, $\rho$                          | m    | 741.3         | 741.3     | 754.1       |
| Acceleration time                               | s    | 4             | 8         | 60          |
| RF harmonic, $h$                                |      | 2310 and 4620 | 4620      | 1113        |
| RF frequency, $f_\mathrm{RF}$                   | MHz  | 200           | 200       | 53.1        |
| RF frequency swing, $\Delta f_\mathrm{RF}$      | kHz  | 143           | 143       | 1.1         |
| RF voltage, $V_\mathrm{RF}$                     | MV   | 20            | Max. 15   | 1           |
| Number of cavities                              |      | $10\ldots12$  | $6$       | $2\times 4$ |
| Longitudinal emittance, $\varepsilon_\mathrm{l}$ | eVs  | 0.45          | 0.45      | 3 to 4      |